# Language Audit

Goal: detect likely language label issues in `questions_flat.csv` by comparing existing `language` vs detected language from `question_text`.


In [ ]:
from pathlib import Path
import re
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW_PATH = ROOT / "data" / "raw" / "questions_flat.csv"
df = pd.read_csv(RAW_PATH, dtype=str, keep_default_na=False)

print(f"Loaded rows: {len(df)}")
print(f"Path: {RAW_PATH}")
display(df[["doc_id", "quiz_id", "quiz_title", "language", "question_text"]].head(3))


In [ ]:
ARABIC_LETTER_RE = re.compile(r"[\u0621-\u064A]")
LATIN_LETTER_RE = re.compile(r"[A-Za-z]")
TOKEN_RE = re.compile(r"[A-Za-z\u00C0-\u00FF']+")

EN_STOPWORDS = {
    "the", "and", "or", "of", "to", "in", "is", "are", "was", "were", "be", "for", "with", "on", "at", "by", "from", "as", "an", "a", "what", "which", "who", "how", "when", "where"
}
FR_STOPWORDS = {
    "le", "la", "les", "de", "des", "du", "et", "ou", "est", "sont", "dans", "pour", "avec", "sur", "par", "comme", "qui", "que", "quoi", "quand", "où", "un", "une", "au", "aux"
}
FR_HINT_CHARS = set("àâçéèêëîïôùûüÿœæ")


def detect_language(text: str) -> tuple[str, float, str]:
    text = str(text or "").strip()
    if not text:
        return ("unknown", 0.0, "empty")

    ar_count = len(ARABIC_LETTER_RE.findall(text))
    lat_count = len(LATIN_LETTER_RE.findall(text))

    # Arabic dominant script check.
    if ar_count >= 5 and ar_count > (lat_count * 1.2):
        confidence = min(0.98, 0.70 + (ar_count / 400.0))
        return ("ar", confidence, "arabic_script_dominant")
    if ar_count > 0 and lat_count == 0:
        return ("ar", 0.85, "arabic_script_only")

    lower = text.lower()
    tokens = TOKEN_RE.findall(lower)
    token_count = max(len(tokens), 1)

    en_hits = sum(token in EN_STOPWORDS for token in tokens)
    fr_hits = sum(token in FR_STOPWORDS for token in tokens)
    en_score = en_hits / token_count
    fr_score = fr_hits / token_count
    has_fr_hint_chars = any(ch in lower for ch in FR_HINT_CHARS)

    if fr_score >= (en_score * 1.2) and (fr_score >= 0.04 or has_fr_hint_chars):
        confidence = min(0.95, 0.55 + (fr_score * 2.0) + (0.10 if has_fr_hint_chars else 0.0))
        return ("fr", confidence, "fr_stopwords_or_chars")
    if en_score > (fr_score * 1.2) and en_score >= 0.04:
        confidence = min(0.95, 0.55 + (en_score * 2.0))
        return ("en", confidence, "en_stopwords")

    if lat_count > 0:
        return ("latin_unknown", 0.35, "latin_without_strong_signal")
    if ar_count > 0:
        return ("ar", 0.60, "arabic_fallback")
    return ("unknown", 0.20, "no_signal")

In [ ]:
# Use quiz_title only for language detection.
text_for_detection = df["quiz_title"].fillna("")

detected = text_for_detection.apply(detect_language)
df[["detected_language", "detect_confidence", "detect_reason"]] = pd.DataFrame(
    detected.tolist(),
    index=df.index,
)

df["language"] = df["language"].str.strip().str.lower()
df["high_confidence"] = (
    df["detected_language"].isin(["en", "fr", "ar"])
    & (df["detect_confidence"] >= 0.60)
)
df["label_mismatch"] = df["high_confidence"] & (df["language"] != df["detected_language"])

print("Rows:", len(df))
print("High-confidence detected rows:", int(df["high_confidence"].sum()))
print("High-confidence mismatches:", int(df["label_mismatch"].sum()))


In [ ]:
print("Existing label distribution")
display(df["language"].value_counts().to_frame("rows"))

print("Detected distribution (high confidence only)")
display(df.loc[df["high_confidence"], "detected_language"].value_counts().to_frame("rows"))

print("Transition table (existing -> detected, high confidence only)")
transition = pd.crosstab(
    df.loc[df["high_confidence"], "language"],
    df.loc[df["high_confidence"], "detected_language"],
)
display(transition)

In [ ]:
mismatch_df = df[df["label_mismatch"]].copy()

print(f"Mismatch rows: {len(mismatch_df)}")
print(f"Affected quizzes: {mismatch_df['quiz_id'].nunique()}")

display(
    mismatch_df.groupby(["language", "detected_language", "detect_reason"]).size()
    .sort_values(ascending=False)
    .rename("rows")
    .reset_index()
)

display(
    mismatch_df.groupby(["quiz_id", "quiz_title", "language", "detected_language"]).size()
    .sort_values(ascending=False)
    .head(20)
    .rename("rows")
    .reset_index()
)

In [ ]:
sample_cols = [
    "doc_id",
    "quiz_id",
    "quiz_title",
    "language",
    "detected_language",
    "detect_confidence",
    "detect_reason",
    "question_text",
]

display(mismatch_df[sample_cols].sort_values(["language", "detected_language", "detect_confidence"], ascending=[True, True, False]).head(50))

In [ ]:
# No file export. Keep analysis in notebook output only.
print("Skipping CSV export by request.")
display(mismatch_df[[
    "doc_id",
    "quiz_id",
    "quiz_title",
    "language",
    "detected_language",
    "detect_confidence",
    "detect_reason",
    "question_text",
]].head(20))
